## Selection correction (winner's curse) for the head-level fidelity map — LOCAL, no GPU

Follow-up to `09_stage2_fidelity_map_kaggle.ipynb` → finding 05. The "best head" number
there = **the maximum over 1024 locations**, so it is necessarily inflated by selection bias.
This notebook runs 3 correction checks, all from the files already downloaded into
`output/09_stage2_fidelity_map_kaggle/`:

1. **Selection and evaluation separated across templates** (from the CSV): pick the head using 3 templates,
   measure on the 4th template that took no part in the choice (4 folds, leave-one-template-out).
2. **Max-statistic permutation test** (from the npz): a null distribution for
   "maximum rho over 1024 heads" — a p-value that ALREADY accounts for selection.
3. **Cell split-half**: pick the head on a random half of the cells, evaluate on the other half (200x).

Bonus: check the **cross-type heads L11 H16 & L18 H14** as FIXED locations (not the result
of per-type selection). Clean results → `notes/findings/06_...`.


In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr

DATA_DIR = "output/09_stage2_fidelity_map_kaggle"
if not os.path.isdir(DATA_DIR):
    DATA_DIR = os.path.join("notebooks", DATA_DIR)  # if run from the repo root
assert os.path.isdir(DATA_DIR), f"data folder not found: {DATA_DIR}"

RANDOM_SEED = 42
N_PERM = 2000        # permutations for the max-statistic test
N_SPLIT = 200        # split-half repetitions

heads_npz = np.load(os.path.join(DATA_DIR, "emb_heads.npz"), allow_pickle=True)
heads_all = heads_npz["emb"]                      # [4, n_g, 32, 32, 128] fp16
GROUP_KEYS = [str(g) for g in heads_npz["group_keys"]]
group_real_dist = np.load(os.path.join(DATA_DIR, "group_real_dist.npy"))
fmap = pd.read_csv(os.path.join(DATA_DIR, "fidelity_map_full.csv"))

N_TEMPLATES, n_g, NUM_LAYERS, NUM_HEADS, HEAD_DIM = heads_all.shape
attr_types = np.array([gk.split(" :: ", 1)[0] for gk in GROUP_KEYS])
TYPES = sorted(set(attr_types.tolist()))
type_idx = {t: np.where(attr_types == t)[0] for t in TYPES}
print(f"{n_g} cells, {len(TYPES)} types, heads {NUM_LAYERS}x{NUM_HEADS}x{HEAD_DIM}, "
      f"{N_TEMPLATES} templates, map {fmap.shape}")


### 0. Prepare the material per type

Per type: the submatrix of real distances between cells (drop cells with NaN pairs so that
the permutation is clean) + cosine distance per head (Tmean) as a [1024, n_pairs] matrix.


In [ ]:
def clean_subset(subset_idx):
    """Drop the cells that cause NaN in the within-type real-distance submatrix."""
    sub = group_real_dist[np.ix_(subset_idx, subset_idx)]
    keep = np.ones(len(subset_idx), dtype=bool)
    while True:
        m = sub[np.ix_(keep, keep)]
        nan_per_cell = np.isnan(m).sum(axis=1)
        if nan_per_cell.max() == 0:
            break
        worst_local = int(np.argmax(nan_per_cell))
        keep_pos = np.where(keep)[0]
        keep[keep_pos[worst_local]] = False
    return subset_idx[keep]

def cosine_dist_per_head(subset_idx, t_tag="Tmean"):
    """[NUM_LAYERS*NUM_HEADS, n_pairs] cosine distance between the subset cells, per head."""
    if t_tag == "Tmean":
        X = heads_all[:, subset_idx].astype(np.float32).mean(axis=0)   # [n, 32, 32, 128]
    else:
        X = heads_all[int(t_tag[1]), subset_idx].astype(np.float32)
    n = len(subset_idx)
    X = X.reshape(n, NUM_LAYERS * NUM_HEADS, HEAD_DIM)
    Xn = X / (np.linalg.norm(X, axis=2, keepdims=True) + 1e-8)
    sims = np.einsum("ahd,bhd->hab", Xn, Xn)                            # [1024, n, n]
    iu = np.triu_indices(n, k=1)
    return 1.0 - sims[:, iu[0], iu[1]]                                  # [1024, n_pairs]

def rank_rows(M):
    """rankdata per row, vectorized (average ties ~ a double argsort is enough on continuous data)."""
    order = np.argsort(M, axis=-1)
    ranks = np.empty_like(order, dtype=np.float64)
    rng_ = np.arange(M.shape[-1], dtype=np.float64)
    np.put_along_axis(ranks, order, np.broadcast_to(rng_, M.shape).copy(), axis=-1)
    return ranks

def spearman_matrix(rep_ranked, real_vec):
    """Spearman between each row of rep_ranked [K, P] and real_vec [P] (ranked here)."""
    r = rankdata(real_vec)
    r = r - r.mean()
    R = rep_ranked - rep_ranked.mean(axis=1, keepdims=True)
    num = R @ r
    den = np.sqrt((R ** 2).sum(axis=1) * (r ** 2).sum()) + 1e-12
    return num / den

material = {}
for ty in TYPES:
    idxs = clean_subset(type_idx[ty])
    dropped = len(type_idx[ty]) - len(idxs)
    sub_real = group_real_dist[np.ix_(idxs, idxs)]
    rep = cosine_dist_per_head(idxs, "Tmean")
    material[ty] = dict(idxs=idxs, real=sub_real, rep=rep, rep_ranked=rank_rows(rep))
    print(f"{ty}: {len(idxs)} cells (dropped {dropped}), {rep.shape[1]} pairs")


### Check 1 — Selection and evaluation separated across templates (from the CSV)

Pick the best head using the mean rho of 3 templates → report that head's rho on the 4th
template that did NOT take part in the choice. Repeated over 4 folds. If the held-out rho stays close to the
Tmean number → the gain is not selection luck.


In [ ]:
ph = fmap[fmap["component"] == "head"].copy()
ph["loc"] = ph["layer"].astype(str) + ":" + ph["head"].astype(str)
piv = {ty: ph[ph["attr_type"] == ty].pivot_table(index="loc", columns="template", values="rho")
       for ty in TYPES}

print("=" * 88)
rows1 = []
for ty in TYPES:
    P = piv[ty]
    naive_resid = fmap[(fmap["attr_type"] == ty) & (fmap["component"] == "resid") &
                       (fmap["template"] == "Tmean")]["rho"].max()
    tmean_max = float(P["Tmean"].max())
    held = []
    for t_out in range(4):
        train_cols = [f"T{t}" for t in range(4) if t != t_out]
        sel_loc = P[train_cols].mean(axis=1).idxmax()
        held.append(float(P.loc[sel_loc, f"T{t_out}"]))
    held = np.array(held)
    rows1.append(dict(attr_type=ty, resid_baseline=float(naive_resid),
                      tmean_max_inflated=tmean_max,
                      heldout_mean=float(held.mean()), heldout_min=float(held.min())))
    print(f"{ty:20s} residual={naive_resid:+.3f}  max-Tmean(inflated)={tmean_max:+.3f}  "
          f"held-out per fold: {np.round(held, 3)}  mean={held.mean():+.3f}")
cek1 = pd.DataFrame(rows1)


### Check 2 — Max-statistic permutation test (full selection correction)

Null hypothesis: there is NO embedding-survey relation AT ALL. Each permutation:
shuffle the cell labels in the real-distance matrix → recompute the **max rho over all 1024 heads**
→ that is the null distribution for the "best head rho" statistic. p = how often the null
max >= the observed max. This automatically accounts for "picking the champion out of 1024".


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
print("=" * 88)
rows2 = []
for ty in TYPES:
    b = material[ty]
    n = len(b["idxs"])
    iu = np.triu_indices(n, k=1)
    real_flat = b["real"][iu]
    obs = spearman_matrix(b["rep_ranked"], real_flat)
    obs_max = float(obs.max())
    null_max = np.empty(N_PERM)
    for k in range(N_PERM):
        perm = rng.permutation(n)
        null_max[k] = spearman_matrix(b["rep_ranked"], b["real"][np.ix_(perm, perm)][iu]).max()
    p_sel = float((null_max >= obs_max).mean())
    rows2.append(dict(attr_type=ty, obs_max=obs_max, null_max_mean=float(null_max.mean()),
                      null_max_p95=float(np.quantile(null_max, 0.95)), p_selection_corrected=p_sel))
    print(f"{ty:20s} max-rho obs={obs_max:+.3f} | null max: mean={null_max.mean():+.3f} "
          f"p95={np.quantile(null_max, 0.95):+.3f} | p(selection-corrected)={p_sel:.4f}")
cek2 = pd.DataFrame(rows2)


### Check 3 — Cell split-half: how stable is the head choice + held-out rho

200x: split the cells of one type into two random halves → pick the best head on half A → measure that
head's rho on half B. Reports the held-out rho distribution (the HONEST number for
"if I use this head on new data, what do I get?").


In [ ]:
rng = np.random.default_rng(RANDOM_SEED + 1)
print("=" * 88)
rows3 = []
for ty in TYPES:
    b = material[ty]
    idxs = b["idxs"]; n = len(idxs)
    full_iu = np.triu_indices(n, k=1)
    pair_pos = {(i, j): k for k, (i, j) in enumerate(zip(full_iu[0], full_iu[1]))}
    held_rhos, chosen = [], []
    for s in range(N_SPLIT):
        perm = rng.permutation(n)
        half = n // 2
        A, B = np.sort(perm[:half]), np.sort(perm[half:])
        if len(A) < 5 or len(B) < 5:
            continue
        iuA = np.triu_indices(len(A), k=1); iuB = np.triu_indices(len(B), k=1)
        posA = np.array([pair_pos[(i, j)] for ii, i in enumerate(A) for j in A[ii+1:]])
        posB = np.array([pair_pos[(i, j)] for ii, i in enumerate(B) for j in B[ii+1:]])
        realA = b["real"][np.ix_(A, A)][iuA]; realB = b["real"][np.ix_(B, B)][iuB]
        rhoA = spearman_matrix(rank_rows(b["rep"][:, posA]), realA)
        best = int(np.argmax(rhoA))
        rhoB = spearman_matrix(rank_rows(b["rep"][best:best+1, posB]), realB)[0]
        held_rhos.append(float(rhoB)); chosen.append(best)
    held_rhos = np.array(held_rhos)
    top_choice = pd.Series(chosen).value_counts().head(3)
    lbl = ", ".join(f"L{c // NUM_HEADS} H{c % NUM_HEADS} ({v}x)" for c, v in top_choice.items())
    rows3.append(dict(attr_type=ty, heldout_median=float(np.median(held_rhos)),
                      heldout_q25=float(np.quantile(held_rhos, 0.25)),
                      heldout_q75=float(np.quantile(held_rhos, 0.75)),
                      head_most_frequent=lbl))
    print(f"{ty:20s} rho held-out: median={np.median(held_rhos):+.3f} "
          f"IQR=[{np.quantile(held_rhos, 0.25):+.3f}, {np.quantile(held_rhos, 0.75):+.3f}] | "
          f"most frequent head: {lbl}")
cek3 = pd.DataFrame(rows3)


### Bonus — cross-type heads as FIXED locations: L11 H16 and L18 H14

The "general head" claim escapes the per-type winner's curse if the location is FIXED in advance
across all types. Here: rho per type at those 2 heads + a permutation p per type
(multiplied by a Bonferroni correction x1024 as the most conservative bound).


In [ ]:
FIXED = [(11, 16), (18, 14)]
rng = np.random.default_rng(RANDOM_SEED + 2)
print("=" * 88)
rows4 = []
for (L, H) in FIXED:
    flat = L * NUM_HEADS + H
    for ty in TYPES:
        b = material[ty]
        n = len(b["idxs"]); iu = np.triu_indices(n, k=1)
        real_flat = b["real"][iu]
        obs = float(spearman_matrix(b["rep_ranked"][flat:flat+1], real_flat)[0])
        null = np.empty(1000)
        for k in range(1000):
            perm = rng.permutation(n)
            null[k] = spearman_matrix(b["rep_ranked"][flat:flat+1],
                                      b["real"][np.ix_(perm, perm)][iu])[0]
        p_raw = float((null >= obs).mean())
        rows4.append(dict(head=f"L{L} H{H}", attr_type=ty, rho=obs, p_raw=p_raw,
                          p_bonf_1024=min(1.0, p_raw * 1024)))
        print(f"L{L} H{H}  {ty:20s} rho={obs:+.3f}  p_raw={p_raw:.4f}  "
              f"p x1024(conservative)={min(1.0, p_raw * 1024):.3f}")
cek4 = pd.DataFrame(rows4)


In [ ]:
OUT = os.path.join(DATA_DIR, "selection_correction")
os.makedirs(OUT, exist_ok=True)
cek1.to_csv(os.path.join(OUT, "check1_heldout_template.csv"), index=False)
cek2.to_csv(os.path.join(OUT, "check2_maxstat_permutation.csv"), index=False)
cek3.to_csv(os.path.join(OUT, "check3_splithalf.csv"), index=False)
cek4.to_csv(os.path.join(OUT, "check4_fixed_heads.csv"), index=False)
print("Saved in", OUT)


### How to read this

- **Check 1**: `heldout_mean` per type = the number you may quote for the "best head"
  (compare it with `tmean_max_inflated` and `resid_baseline`).
- **Check 2**: `p_selection_corrected` < 0.05 = the best head's edge is NOT the luck of
  picking a champion out of 1024. `null_max_p95` = how high a maximum rho pure noise can
  reach — the most important comparison for the finding 05 numbers.
- **Check 3**: the median held-out rho = the honest expectation of the chosen head's performance on new data;
  `head_most_frequent` = whether the choice is consistent (the same head keeps winning) or a
  lottery (different on every split).
- **Bonus**: if L11 H16 is significant in many types AFTER x1024 → the "general demographic
  geometry head" claim is strong.

Results → see `docs/research-log.md` (Act 2).
